# Fase 4: Benchmarking Internacional

## Comparación con Medellín, Santiago, CDMX y Bogotá

### Objetivos:
1. Compilar métricas de sistemas comparables
2. Posicionar a Lima en el contexto latinoamericano
3. Identificar factores de éxito

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from benchmarking import (
    build_benchmark_df, print_benchmark_report,
    plot_ridership_intensity, plot_bc_comparison,
    plot_cost_per_km, plot_scatter
)
import matplotlib.pyplot as plt
import pandas as pd

---
## Datos de Benchmarking

In [ ]:
df = build_benchmark_df()
display_cols = ['city', 'population_M', 'metro_km', 'daily_pax_K', 'pax_per_km',
                'cost_per_km_USD_M', 'bc_ratio', 'fare_integration']
df[display_cols].style.background_gradient(cmap='viridis', subset=['pax_per_km', 'bc_ratio'])

---
## Reporte de Benchmarking

In [ ]:
print_benchmark_report(df)

---
## Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Intensidad
ax = axes[0, 0]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#4CAF50', '#E91E63', 'red']
plot_df = df[df['city'] != 'Lima (propuesta)']
vals = plot_df['pax_per_km'].tolist() + [df[df['city']=='Lima (propuesta)']['pax_per_km'].iloc[0]]
lbls = plot_df['city'].tolist() + ['Lima\n(prop.)']
bars = ax.bar(range(len(vals)), vals, color=colors, alpha=0.8)
ax.axhline(y=df[df['city']=='Lima (actual)']['pax_per_km'].iloc[0],
           color='#E91E63', linestyle='--', alpha=0.5, label='Lima actual')
ax.set_xticks(range(len(vals))); ax.set_xticklabels(lbls, fontsize=8)
ax.set_ylabel('Pax/km/día'); ax.set_title('Intensidad de Uso'); ax.legend(fontsize=7)

# 2. B/C
ax = axes[0, 1]
vals = df['bc_ratio'].tolist()
lbls = df['city'].tolist()
ax.bar(range(len(vals)), vals, color=colors, alpha=0.8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='B/C = 1')
ax.set_xticks(range(len(vals))); ax.set_xticklabels(lbls, fontsize=8)
ax.set_ylabel('B/C'); ax.set_title('Costo-Beneficio'); ax.legend(fontsize=7)

# 3. Costo/km
ax = axes[1, 0]
ax.bar(range(len(vals)), df['cost_per_km_USD_M'].tolist(), color=colors, alpha=0.8)
ax.set_xticks(range(len(vals))); ax.set_xticklabels(df['city'].tolist(), fontsize=8)
ax.set_ylabel('US$M/km'); ax.set_title('Costo de Construcción por km')

# 4. Dispersión
ax = axes[1, 1]
plot_df = df[df['city'] != 'Lima (propuesta)']
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.scatter(row['population_M'], row['pax_per_km'],
               s=row['daily_pax_K']/50, c=colors[i], alpha=0.7,
               edgecolors='black', linewidth=0.5,
               label=f"{row['city']} ({row['daily_pax_K']:.0f}K)")
lp = df[df['city'] == 'Lima (propuesta)'].iloc[0]
ax.scatter(lp['population_M'], lp['pax_per_km'],
           s=lp['daily_pax_K']/50, c='red', alpha=0.7,
           edgecolors='black', linewidth=1.5, marker='s',
           label=f"Lima prop. ({lp['daily_pax_K']:.0f}K)")
ax.set_xlabel('Población (M)'); ax.set_ylabel('Pax/km/día')
ax.set_title('Población vs Intensidad'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## Resumen Fase 4

| Ciudad | Pax/km/día | B/C | Costo/km (US$M) | Integración tarifaria |
|--------|:----------:|:---:|:---------------:|:--------------------:|
| Medellín | ~16K | 1.4 | 60 | Full |
| Santiago | ~17K | 1.2 | 120 | Full |
| CDMX | ~20K | 1.8 | 80 | Parcial |
| Bogotá (BRT) | ~21K | 2.1 | 15 | Full |
| **Lima actual** | **~19K** | **0.9** | **85** | **Ninguna** |
| **Lima propuesta** | **~2.6K** | **0.16** | **73** | **Propuesta** |

**Conclusión:** La red propuesta es demasiado extensa para la demanda real. Se recomienda priorizar líneas de alta densidad (L3, Tren Ica) y establecer integración tarifaria antes de expandir.